In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — TEST DU MODE « FLEX » SUR UN MATCH COMPLET EN REPLAY — réglage du 14/11/2026
# Réglage testé (décidé le 14/11 d'après le run Flex du 13/11 et le test de délai du 14/11) :
#   • Flex seulement pour les 3 GROSSES VAGUES : coups d'envoi, classification, équipes Q1/Q2 (vague_flex) ;
#     second regard, escalade, tenues, arbitrage Q3, scoring et bornes restent au tarif standard ;
#   • attente maximale 2 h 15 par question Flex, puis bascule au tarif standard ;
#   • 128 questions à la fois.
# CRITÈRES FIXÉS AVANT (référence : replay standard du 13/11, Minerois Ster : 5/5 buts, 2 faux, équipes 5 justes / 0 faux) :
#   1. aucun vrai but perdu ; faux buts ≤ référence + 1 ; 0 attribution fausse ;
#   2. durée totale du match < 10 h ;
#   3. au moins 80 % des questions des grosses vagues facturées en Flex (ligne « part Flex »).
# Coût prévu ≈ 1,2 $. Un seul match : Kaggle coupe à 12 h.
# Nécessite sur GitHub : ai/gemini_validator.py, pipeline.py et ai/attribution_equipes.py du 14/11.
# Entrées : codeprojectai (caches), la vidéo de Minerois Ster, mziparameter, test_resolution_gemini_cellule.py.
# Pas de GPU (Accelerator : None). Save & Run All (Commit).
# ════════════════════════════════════════════════════════════════════════════════════════════
TIER = "flex"
MATCHS_A_TESTER = ["minerois_ster"]    # un seul match : avec des vagues Flex de 2 h 30 au plus, ≈ 7 à 8 h
RESOLUTION = "medium"            # validation des coups d'envoi
RESOLUTION_CLASSIF = "low"       # classification des temps forts
REFERENCE = {"minerois_ster": (5, 2)}   # replay standard 128 du 13/11
_REFERENCE_ANCIENNE = {   # (vrais buts, faux buts) : résolution NORMALE quand connue (spa, minerois_ster, franchimont, raeren),
    # sinon run « medium » seul du 08/11 (andrimont : cache arrêté à 61:39, 5 buts évaluables)
    "spa": (7, 0), "minerois_ster": (5, 1), "franchimont": (5, 4), "raeren": (8, 4), "andrimont": (5, 1),
    "p1_minerois": (3, 1), "goe": (2, 5), "stembert": (3, 3), "wanze": (2, 3), "juprelle": (1, 1)}

import glob, io, re, sys, time, contextlib

_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))
                 + glob.glob("/kaggle/working/test_resolution_gemini_cellule.py")), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (l'ajouter aux entrées ou dans /kaggle/working)")
CODE = open(_src, encoding="utf-8").read()
# on retire l'étape tenues + attribution (de « DÉTECTION DES TENUES » jusqu'au calcul du coût réel)
_a = CODE.index('print("\\n" + "=" * 60 + f"\\nDÉTECTION DES TENUES')
_b = CODE.index("# ── coût réel de ce run")
_VERIF_ATTR = '''
# ── 09/11 : vérification de l'ATTRIBUTION FAITE PAR LE PIPELINE (plus par le notebook) ──
import re as _re_v, collections as _col_v
_att_v = (result or {}).get("attribution_equipes") or {}
_ten_v = _att_v.get("tenues") or {}
def _parse_v(t):
    m = _re_v.match(r"maillot (.*) / short (.*)", str(t or ""))
    return (m.group(1), m.group(2)) if m else ("", "")
_TD_v = (_parse_v(_ten_v.get("A")), _parse_v(_ten_v.get("B")))
_mots_v = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
_sc_v = lambda det, vrai: len(_mots_v(det[0]) & _mots_v(vrai[0])) * 2 + len(_mots_v(det[1]) & _mots_v(vrai[1]))
_VT_v = VRAIES_TENUES_TOUTES[MATCH]; _VE_v = VERITE_EQUIPES[MATCH]
_d_v = _sc_v(_TD_v[0], _VT_v[0]) + _sc_v(_TD_v[1], _VT_v[1]); _i_v = _sc_v(_TD_v[0], _VT_v[1]) + _sc_v(_TD_v[1], _VT_v[0])
_map_v = None if _d_v == _i_v else ({"A": "A", "B": "B"} if _d_v > _i_v else {"A": "B", "B": "A"})
_bil_v = _col_v.Counter(); _vu_v = set()
for _b_v in (result or {}).get("buts", []):
    _g_v = float(_b_v.get("time_abs") or 0)
    _m_v = [h for h in _VE_v if abs(int(h.split(":")[0]) * 60 + int(h.split(":")[1]) - _g_v) <= 20 and h not in _vu_v]
    if not _m_v:
        continue
    _vu_v.add(_m_v[0])
    _e_v = _b_v.get("equipe")
    _ok_v = "—" if not _e_v else ("?" if not _map_v else ("juste" if _map_v[_e_v] == _VE_v[_m_v[0]] else "FAUX"))
    _bil_v[(_b_v.get("niveau_equipe") or "non attribué", _ok_v)] += 1
print(f"ATTRIBUTION PIPELINE : tenues {'OK' if _map_v else 'CORRESPONDANCE IMPOSSIBLE'} | "
      f"justes {_bil_v[('confirmé', 'juste')] + _bil_v[('probable', 'juste')]} | "
      f"FAUX {sum(v for k, v in _bil_v.items() if k[1] == 'FAUX')} | "
      f"non attribués {sum(v for k, v in _bil_v.items() if k[0] == 'non attribué')} | "
      f"coût {_att_v.get('cout_dollars', 0)} $ | escalades {_att_v.get('escalades', 0)} | "
      f"recompressées {_att_v.get('requetes_recompressees', 0)} | ÉCHECS {_att_v.get('questions_en_echec', 0)}")
'''
CODE = CODE[:_a] + "\n" + _VERIF_ATTR + "\n" + CODE[_b:]
CODE = re.sub(r'^RESOLUTION = .*$', f'RESOLUTION = "{RESOLUTION}"', CODE, count=1, flags=re.M)
CODE = CODE.replace('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION',
                    'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_KO"] = "{RESOLUTION}"\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_CLASSIF"] = "{RESOLUTION_CLASSIF}"\n'
                    'if "haute_resolution" not in open(_gv, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ gemini_validator.py du 08/11 (2) absent du dépôt GitHub")\n'
                    '_pp = next(iter(_g_res.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)), None)\n'
                    'if not _pp or "[ESCALADE]" not in open(_pp, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ pipeline.py du 08/11 (escalade) absent du dépôt GitHub")\n'
                    'if not _g_res.glob("/kaggle/working/ProjectAIMika/**/attribution_equipes.py", recursive=True) or '
                    '"Step 11b" not in open(_pp, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ ai/attribution_equipes.py et pipeline.py du 09/11 absents du dépôt GitHub")', 1)
assert "SCOUTIA_MEDIA_RESOLUTION_CLASSIF" in CODE, "point d'insertion introuvable"
CODE = f'import os as _os_tier\n_os_tier.environ["SCOUTIA_GEMINI_TIER"] = "{TIER}"\n' + CODE
# 13/11 : au-delà de 30 min d'attente, une question Flex part au tarif standard (les quelques « traînardes » du run du
# 13/11 ont retenu la vague des coups d'envoi plus de 3 h 30 ; voir ATTENTE_FLEX_MAX_S).
ATTENTE_FLEX_MAX_S = 8100   # 14/11 : 2 h 15 (1800 était trop court : 1 % des réponses Flex du 13/11 avant 30 min)
CODE = f'_os_tier.environ["SCOUTIA_FLEX_ATTENTE_MAX_S"] = "{ATTENTE_FLEX_MAX_S}"\n' + CODE
CODE = 'import os as _os_tier\n' + CODE
# 13/11 : la cellule de test impose GEMINI_MAX_PARALLEL = 4 (réglage standard) : en Flex, 4 questions à la fois ont
# donné 100 validations en ≈ 8 h 20 (run du 12/11). On pose ici PARALLELE_FLEX questions à la fois.
PARALLELE_FLEX = 128     # 13/11 : les 100 validations de coups d'envoi partent toutes d'un coup (maximum autorisé : 128)
import re as _re_par
CODE, _n_par = _re_par.subn(r'os\.environ\["GEMINI_MAX_PARALLEL"\] = "\d+"', f'os.environ["GEMINI_MAX_PARALLEL"] = "{PARALLELE_FLEX}"', CODE)
assert _n_par == 1, "réglage GEMINI_MAX_PARALLEL introuvable dans la cellule"
# 12/11 : le run du 12/11 a échoué car la version de google-genai installée par le dépôt ne connaissait pas
# « service_tier » (toutes les questions refusées AVANT envoi, 0 $ payé, 0 but). Le test de compatibilité avait réussi
# avec google-genai 3.2.0 : on l'installe AVANT le chargement du pipeline. (À reporter dans requirements.txt.)
_MAJ_GENAI = '''
import subprocess as _sp_g, sys as _sys_g
_sp_g.run([_sys_g.executable, "-m", "pip", "install", "-q", "-U", "google-genai>=3.2.0"], check=False)
'''
assert CODE.count("from pipeline import run_pipeline") == 1, "point d'insertion (import du pipeline) introuvable"
CODE = CODE.replace("from pipeline import run_pipeline", _MAJ_GENAI + "from pipeline import run_pipeline", 1)   # après les dépendances du dépôt, avant le chargement de google-genai
CODE = CODE.replace("/ 1e6 * 0.75", "/ 1e6 * 0.375").replace("/ 1e6 * 3.75", "/ 1e6 * 1.875")   # tarif Flex
_CONTROLE_TIER = '''
_src_gv_t = open(_gv, encoding="utf-8").read()
if "SCOUTIA_GEMINI_TIER" not in _src_gv_t or "def vague_flex" not in _src_gv_t:
    raise SystemExit("❌ ai/gemini_validator.py du 14/11 (Flex par grosses vagues) absent du dépôt GitHub — rien n'est payé")
_pl_t = next(iter(_g_res.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)), None)
_at_t = next(iter(_g_res.glob("/kaggle/working/ProjectAIMika/**/attribution_equipes.py", recursive=True)), None)
if not _pl_t or "_vague_flex(_ko_call)" not in open(_pl_t, encoding="utf-8").read() or not _at_t \
        or "_vf(dem)" not in open(_at_t, encoding="utf-8").read():
    raise SystemExit("❌ pipeline.py ou ai/attribution_equipes.py du 14/11 absents du dépôt GitHub — rien n'est payé")
'''
CODE = CODE.replace('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION', _CONTROLE_TIER + 'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION', 1)


import subprocess
# Chaque match tourne dans un PROCESSUS PYTHON SÉPARÉ : le cache Gemini, ses fichiers de sortie et les modules du
# pipeline repartent de zéro à chaque match (sinon l'état du 1er match se mélangerait aux suivants).
BILAN = {}
for m in MATCHS_A_TESTER:
    t0 = time.time()
    print(f"\n{'#' * 90}\n# MATCH {m} — résolution {RESOLUTION}\n{'#' * 90}", flush=True)
    code_m = re.sub(r'^MATCH = .*$', f'MATCH = "{m}"', CODE, count=1, flags=re.M)
    f = f"/kaggle/working/_test_resolution_{m}.py"
    open(f, "w", encoding="utf-8").write(code_m)
    proc = subprocess.Popen([sys.executable, "-u", f], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    lignes = []
    for l in proc.stdout:
        sys.stdout.write(l); lignes.append(l)
    proc.wait()
    sortie = "".join(lignes)
    _t = re.search(r"Niveaux de service des questions payées : (\{[^}]*\})", sortie)
    if "FLEX IMPOSSIBLE" in sortie or "validation error for GenerateContentConfig" in sortie:
        raise SystemExit(f"❌ {m} : Flex n'a pas fonctionné (version de google-genai) — test arrêté, voir le journal")
    print(f"   ⏱ {m} : durée totale {(time.time() - t0) / 60:.0f} min | niveaux de service : {_t.group(1) if _t else 'non affichés'}")
    try:
        import ast as _ast_t
        _niv = _ast_t.literal_eval(_t.group(1)) if _t else {}
        _fl = _niv.get("flex", 0); _rp = _niv.get("standard (repli après attente Flex)", 0)
        print(f"   part Flex des grosses vagues : {_fl} / {_fl + _rp} = {100 * _fl / max(1, _fl + _rp):.0f} % "
              f"(critère ≥ 80 %) | petites vagues au standard : {_niv.get('standard (petite vague)', 0)}"
              f" | durée {(time.time() - t0) / 60:.0f} min (critère < 600 min)")
    except Exception as _e_niv:
        print(f"   part Flex non calculée : {_e_niv}")
    b = re.search(r"BILAN DÉTECTION : (\d+)/(\d+) vrais buts \| (\d+) faux", sortie)
    c = re.search(r"TOTAL ≈ ([\d.]+) \$", sortie)
    manques = re.findall(r"but réel (\S+) : ❌ MANQUÉ", sortie)
    esc = re.search(r"\[ESCALADE\] (\d+) cas", sortie)
    att = re.search(r"ATTRIBUTION PIPELINE : tenues (\S+).*?justes (\d+) \| FAUX (\d+) \| non attribués (\d+) \| coût ([\d.]+) \$", sortie)
    changes = re.findall(r"\[ESCALADE\] (\S+) \(classification [^)]*\) : résolution réduite (\S+) → normale (\S+)", sortie)
    BILAN[m] = dict(trouves=int(b.group(1)) if b else None, vrais=int(b.group(2)) if b else None,
                    faux=int(b.group(3)) if b else None, cout=float(c.group(1)) if c else None,
                    manques=manques, minutes=(time.time() - t0) / 60, esc=int(esc.group(1)) if esc else 0,
                    changes=[f"{t} {a}→{b}" for t, a, b in changes if a != b],
                    att=(att.group(1), int(att.group(2)), int(att.group(3)), int(att.group(4)), float(att.group(5))) if att else None)

print(f"\n{'═' * 90}\nBILAN — résolution {RESOLUTION} sur {len(BILAN)} match(s)\n{'═' * 90}")
print("   match           buts trouvés   faux   référence (trouvés, faux)   coût     durée   escalades  manqués")
tot_t = tot_v = tot_c = 0; perdus = []; trop_faux = []
for m, r in BILAN.items():
    ref = REFERENCE.get(m, (None, None))
    if r["trouves"] is None:
        print(f"   {m:15s} — résultat non lu (voir la sortie du match)"); continue
    tot_t += r["trouves"]; tot_v += r["vrais"]; tot_c += r["cout"] or 0
    if ref[0] is not None and r["trouves"] < ref[0]:
        perdus.append(m)
    if ref[0] is None and r["trouves"] < r["vrais"]:
        perdus.append(m + " (à vérifier : pas de référence récente)")
    if ref[1] is not None and r["faux"] > ref[1] + 1:
        trop_faux.append(m)
    print(f"   {m:15s} {r['trouves']:>3d} / {r['vrais']:<3d}       {r['faux']:>3d}    {str(ref):26s}  "
          f"{(r['cout'] or 0):5.2f} $  {r['minutes']:4.0f} min  {r['esc']:5d}      {', '.join(r['manques'])}"
          + (f"  | changés : {', '.join(r['changes'])}" if r['changes'] else ""))
n_esc = sum(r["esc"] for r in BILAN.values())
_at = [r["att"] for r in BILAN.values() if r.get("att")]
print(f"\n   ATTRIBUTION (faite par le pipeline) : tenues OK {sum(1 for a in _at if a[0] == 'OK')}/{len(_at)} | "
      f"justes {sum(a[1] for a in _at)} | FAUX {sum(a[2] for a in _at)} | non attribués {sum(a[3] for a in _at)} | "
      f"coût {sum(a[4] for a in _at):.2f} $  (référence : normale, 41 justes / 0 faux / 2 non attribués sur 43 vrais buts)")
print("   CRITÈRES ATTRIBUTION (fixés le 11/11) : FAUX = 0 | non attribués ≤ 4 (sur les vrais buts trouvés) | "
      "aucune ligne « ÉCHECS » > 0 dans le détail par match (sinon run non valable pour l'attribution)")
print(f"\n   TOTAL : {tot_t} / {tot_v} vrais buts | coût Gemini mesuré ≈ {tot_c:.2f} $ (sans l'attribution des équipes)")
print(f"   ⚠️ les {n_esc} escalades en résolution normale ont pu être relues GRATUITEMENT dans l'ancien cache : en production,"
      f" compter ≈ {n_esc * 0.017:.2f} $ de plus au total (≈ 0,017 $ par question KO en résolution normale)")
print(f"   matchs avec un vrai but de moins que la référence : {perdus or 'aucun'}")
print(f"   matchs avec plus d'1 faux but de plus que la référence : {trop_faux or 'aucun'}")
ok = not [p for p in perdus if 'à vérifier' not in p] and not trop_faux
print(f"   → {'RÉUSSI : la résolution « medium » peut devenir le réglage par défaut' if ok else 'ÉCHEC ou à examiner : voir les matchs listés'}")


##########################################################################################
# MATCH minerois_ster — résolution medium
##########################################################################################
  Clés lues dans /kaggle/input/datasets/michaelziant/mziparameter/param.env : SECRET_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, STRIPE_SECRET_KEY, STRIPE_PUBLIC_KEY, R2_SECRET_ACCESS_KEY, RUNPOD_API_KEY

ÉTAPE 1/3 — Clone
Cloning into '/kaggle/working/ProjectAIMika'...
Clone OK

ÉTAPE 2/3 — Dépendances
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.3 MB/s eta 0:00:00
   ━